In [1]:
import numpy as np
import xarray as xr
import os

This code is based on ECMWF algorithm described here 
https://confluence.ecmwf.int/display/CKB/Near+surface+meteorological+variables+from+1979+to+2019+derived+from+bias-corrected+reanalysis+%28WFDE5%29%3A+Product+User+Guide#Nearsurfacemeteorologicalvariablesfrom1979to2019derivedfrombiascorrectedreanalysis(WFDE5):ProductUserGuide-references9

Originally this algorithm is based on Buck (1981) paper (formula 4a) \
Buck, A. L., 1981: New Equations for Computing Vapor Pressure and Enhancement Factor. J. Appl. Meteor. Climatol., 20, 1527–1532, https://doi.org/10.1175/1520-0450(1981)020<1527:NEFCVP>2.0.CO;2.

In [2]:
# Constants
EAw = 6.1121
EBw = 18.729
ECw = 257.87
EDw = 227.3

EAi = 6.1115
EBi = 23.036
ECi = 279.82
EDi = 333.7

FAw = 1.000072
FBw = 3.2e-6
FCw = 5.9e-10

FAi = 1.000022
FBi = 3.83e-6
FCi = 6.4e-10

In [3]:
def Fw(T, P):
    '''
    Computing enhancement factor (water)
    '''
    return P * (FCw * T**2 + FBw) + FAw

def Fi(T, P):
    '''
    Computing enhancement factor (ice)
    '''
    return P * (FCi * T**2 + FBi) + FAi

def compute_SVP_ES(t2m, sp):
    '''
    Input: 
    t2m - temperature at 2 m [ºC]
    sp - surface pressure [hPa]

    Output:
    svp - saturated vapout pressure
    '''

    t2m_c = t2m - 273.15  # Convert from K to °C
    sp_hpa = sp / 100     # Convert Pa to hPa
    
    mask_water = t2m_c >= 0
    mask_ice = ~mask_water

    SVP = xr.zeros_like(t2m_c)

    # Water
    SVP_water = (
        EAw * np.exp((EBw - t2m_c / EDw)
        * t2m_c / (t2m_c + ECw))
        * Fw(t2m_c, sp_hpa)
    )
    # Ice
    SVP_ice = (
        EAi * np.exp((EBi - t2m_c / EDi)
        * t2m_c / (t2m_c + ECi))
        * Fi(t2m_c, sp_hpa)
    )

    SVP = xr.where(mask_water, SVP_water, SVP_ice)
    return SVP

def compute_VP_ES(d2m, sp):
    '''
    Input:
    t2m - temperature at 2 m [ºC]
    sp - surface pressure [hPa]

    Output:
    vp - vapour pressure
    '''
    d2m_c = d2m - 273.15  # Convert from K to °C
    sp_hpa = sp / 100     # Convert Pa to hPa
    return (
        EAw * np.exp((EBw - d2m_c / EDw)
        * d2m_c / (d2m_c + ECw))
        * Fw(d2m_c, sp_hpa)
    )

def compute_water_content(svp, vp, sp):
    """
    Input:
    svp, vp -  saturated and actual vapor pressure [hPa]
    sp: surface pressure [Pa]
    
    Output:
    swc - saturated water content 
    wc - water content 
    rh - relative humidity
    """

    # Constants
    A = 0.37802
    B = 0.62198

    sp_hpa = sp / 100     # Convert Pa to hPa

    swc = svp / (sp_hpa - svp * A) * B
    wc  = vp  / (sp_hpa - vp * A) * B

    rh = (wc / swc) * 100
    return swc, wc, rh


def process_year(year, base_path='/glade/campaign/cgd/oce/people/rita/ERA5/', chunks={'time': 24, 'lat': 160, 'lon': 160}):
    path = os.path.join(base_path, str(year))
    print(f"Processing {year}...")

    d2m = xr.open_dataset(f'{path}/D2M_{year}.nc')['D2M'].chunk(chunks)
    t2m = xr.open_dataset(f'{path}/T2M_{year}.nc')['T2M'].chunk(chunks)
    sp  = xr.open_dataset(f'{path}/SP_{year}.nc')['SP'].chunk(chunks)

    svp = compute_SVP_ES(t2m, sp)
    vp  = compute_VP_ES(d2m, sp)
    swc, q2m, rh = compute_water_content(svp, vp, sp)

    q2m.name = 'Q2M'
    q2m.attrs['units'] = 'kg/kg'
    
    out_path = f'{path}/Q2M_{year}.nc'
    q2m.to_dataset().to_netcdf(out_path, compute=True)
    print(f"Saved to {out_path}")



In [ ]:
for year in range(1941, 2021):
    process_year(year)

Processing 1941...
Saved to /glade/campaign/cgd/oce/people/rita/ERA5/1941/Q2M_1941.nc
Processing 1942...
Saved to /glade/campaign/cgd/oce/people/rita/ERA5/1942/Q2M_1942.nc
Processing 1943...
Saved to /glade/campaign/cgd/oce/people/rita/ERA5/1943/Q2M_1943.nc
Processing 1944...
Saved to /glade/campaign/cgd/oce/people/rita/ERA5/1944/Q2M_1944.nc
Processing 1945...


In [ ]:
# OLD but correct

# %%time

# # YEAR = 1940
# PATH = '/glade/campaign/cgd/oce/people/rita/ERA5/' + str(YEAR) + '/'
# chunks = {'time': 24, 'lat': 160, 'lon': 160}  # Fit in 4 GB

# d2m = xr.open_dataset(PATH + 'D2M_' + str(YEAR) +'.nc')#,chunks=chunks)['D2M'] 
# t2m = xr.open_dataset(PATH + 'T2M_' + str(YEAR) +'.nc')#,chunks=chunks)['T2M']
# sp = xr.open_dataset(PATH + 'SP_' + str(YEAR) +'.nc')#,chunks=chunks)['SP'] 


# d2m = d2m['D2M'].chunk(chunks) 
# t2m = t2m['T2M'].chunk(chunks)  
# sp = sp['SP'].chunk(chunks)  

# # %%time 

# # svp = compute_SVP_ES(t2m, sp)
# # vp = compute_VP_ES(d2m, sp)

# # swc, q2m, rh = compute_water_content(svp, vp, sp)

# # q2m.name = 'Q2M'
# # q2m.attrs['units'] = 'kg/kg'

# # %%time
# # q2m.to_dataset().to_netcdf(f'{PATH}Q2M_{YEAR}.nc', compute=True)